# 8.25 — Text classification & sentiment

Text classification listens to many token-level clues, then commits to one document-level label. Classification turns a review, post, ticket, or message into a label another system can act on, using bag-of-words evidence and logistic regression probabilities. Save a copy to Drive to edit.

In [ ]:

import math
import random
import sqlite3
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8025)
np.random.seed(8025)


def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def cosine(a, b):
    a_arr = np.array(a, dtype=float)
    b_arr = np.array(b, dtype=float)
    denom = np.linalg.norm(a_arr) * np.linalg.norm(b_arr)
    if denom == 0:
        return 0.0
    return float(np.dot(a_arr, b_arr) / denom)


def accuracy_score(y_true, y_pred):
    correct = sum(int(a == b) for a, b in zip(y_true, y_pred))
    return correct / max(1, len(y_true))


def precision_recall_f1(y_true, y_pred):
    tp = sum(int(a == 1 and b == 1) for a, b in zip(y_true, y_pred))
    fp = sum(int(a == 0 and b == 1) for a, b in zip(y_true, y_pred))
    fn = sum(int(a == 1 and b == 0) for a, b in zip(y_true, y_pred))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return precision, recall, f1

VOCAB = ["good", "bad", "not_good", "excellent", "awful", "slow", "helpful", "broken", "resolved", "confusing", "love", "hate"]
WEIGHTS = {
    "good": 1.2,
    "bad": -1.5,
    "not_good": -2.0,
    "excellent": 1.6,
    "awful": -1.8,
    "slow": -0.9,
    "helpful": 1.0,
    "broken": -1.3,
    "resolved": 1.1,
    "confusing": -0.8,
    "love": 1.4,
    "hate": -1.6,
}


def sentiment_tokens(text):
    lowered = text.lower().replace(".", " ").replace(",", " ").replace("!", " ")
    raw = lowered.split()
    tokens = []
    skip_next = False
    for i, token in enumerate(raw):
        if skip_next:
            skip_next = False
            continue
        if token == "not" and i + 1 < len(raw) and raw[i + 1] == "good":
            tokens.append("not_good")
            skip_next = True
        else:
            tokens.append(token)
    return tokens


def vectorize_sentiment(text, vocab=VOCAB):
    counts = Counter(sentiment_tokens(text))
    return np.array([counts[word] for word in vocab], dtype=float)


def sentiment_logreg(x, w, b=0.0, tau=0.5):
    z = float(np.dot(x, w) + b)
    p = sigmoid(z)
    pred = int(p > tau)
    return z, p, pred


def sentiment_dataset_ladder():
    d1 = [
        {"text": "good good", "label": 1},
        {"text": "bad", "label": 0},
        {"text": "not good", "label": 0},
    ]
    d2 = [
        {"text": "excellent helpful resolved", "label": 1},
        {"text": "good helpful", "label": 1},
        {"text": "bad broken", "label": 0},
        {"text": "awful slow", "label": 0},
        {"text": "love good service", "label": 1},
    ]
    d3 = d2 + [
        {"text": "not good slow", "label": 0},
        {"text": "good but confusing", "label": 0},
        {"text": "resolved but slow", "label": 1},
        {"text": "not good and broken", "label": 0},
    ]
    d4 = d3 + [
        {"text": "The setup was confusing but support resolved it and the result is good", "label": 1},
        {"text": "I love the helpful checklist but the upload is slow", "label": 1},
        {"text": "The workflow is broken and not good for launch", "label": 0},
        {"text": "Excellent guidance turned a bad draft into a good campaign", "label": 1},
    ]
    d5 = d4 + [
        {"text": "After two retries the dashboard finally resolved the issue, but the first experience was slow and confusing", "label": 1},
        {"text": "The copy looks good in isolation, not good once the broken approval flow delays every review", "label": 0},
        {"text": "Excellent onboarding and helpful examples made the difficult migration feel good", "label": 1},
        {"text": "I wanted to love it, but awful latency and broken exports made the release not good", "label": 0},
        {"text": "Support resolved the bad import and the final report is helpful", "label": 1},
    ]
    return [
        {"rung": "D1", "name": "lesson good bad not-good toy", "items": d1},
        {"rung": "D2", "name": "five clean labeled reviews", "items": d2},
        {"rung": "D3", "name": "negation and threshold imbalance", "items": d3},
        {"rung": "D4", "name": "inline sentiment snippets", "items": d4},
        {"rung": "D5", "name": "longer mixed-domain reviews", "items": d5},
    ]


def sentiment_weight_vector(vocab=VOCAB):
    return np.array([WEIGHTS[word] for word in vocab], dtype=float)


## The concept, built once: weighted sentiment evidence

The lesson's classifier uses a feature vector and a thresholded sigmoid:

$$z=x^	op w+b,\quad P(y=1\mid x)=\sigma(z)=rac{1}{1+e^{-z}},\quad \hat y=\mathbb{1}[P(y=1\mid x)>	au].$$

We first implement the reusable scorer and assert the lesson numbers: `[good,bad]`, `x=[2,0]`, `w=[1.2,-1.5]`, and logit `2.4`.

In [ ]:

lesson_x = np.array([2.0, 0.0])
lesson_w = np.array([1.2, -1.5])
lesson_z, lesson_p, lesson_pred = sentiment_logreg(lesson_x, lesson_w, b=0.0, tau=0.9)

assert abs(lesson_z - 2.4) < 1e-12
assert round(lesson_p, 3) == 0.917
assert lesson_p > 0.9
assert lesson_pred == 1

print("lesson logit", lesson_z)
print("lesson probability", round(lesson_p, 3))
print("threshold 0.9 prediction", lesson_pred)


## Composition and thresholds are part of the method

The lesson contrasts `good` with `not good`, computes the confusion-matrix accuracy `17/20=0.85`, and shows that thresholds control operating points rather than living forever at `0.5`.

In [ ]:

good_prob = sigmoid(1.2)
not_good_prob = sigmoid(-0.8)
confusion = np.array([[8, 2], [1, 9]])
accuracy = np.trace(confusion) / confusion.sum()
probs = np.array([0.1, 0.4, 0.6, 0.9])
labels = np.array([0, 0, 1, 1])
preds = (probs > 0.5).astype(int)

assert round(good_prob, 3) == 0.769
assert round(not_good_prob, 3) == 0.310
assert abs(accuracy - 0.85) < 1e-12
assert accuracy_score(labels, preds) == 1.0

print("good vs not good", round(good_prob, 3), round(not_good_prob, 3))
print("confusion accuracy", accuracy)
print("toy threshold predictions", preds.tolist())


## The dataset ladder: D1 to D5 synthetic text

Family F7 is built inline. Each rung adds realistic text complexity: clean words, negation, snippets, and longer mixed-domain reviews.

In [ ]:

ladder = sentiment_dataset_ladder()

for rung in ladder:
    labels = [item["label"] for item in rung["items"]]
    positives = sum(labels)
    negatives = len(labels) - positives
    print(rung["rung"], rung["name"], "n=", len(labels), "pos=", positives, "neg=", negatives)
    print("sample:", rung["items"][0]["text"])


## Run the same logistic method across D1-D5

The metric is classification accuracy, with F1 printed as a threshold sanity companion for imbalanced sentiment errors.

In [ ]:

w = sentiment_weight_vector()
results = []

for rung in ladder:
    y_true = []
    y_pred = []
    scores = []
    for item in rung["items"]:
        x = vectorize_sentiment(item["text"])
        z, p, pred = sentiment_logreg(x, w, b=0.0, tau=0.5)
        y_true.append(item["label"])
        y_pred.append(pred)
        scores.append(z)
    precision, recall, f1 = precision_recall_f1(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    results.append({"rung": rung["rung"], "accuracy": acc, "f1": f1, "scores": scores, "preds": y_pred})

print("rung accuracy f1")
for row in results:
    print(row["rung"], round(row["accuracy"], 3), round(row["f1"], 3))


## Results visualization: evidence panels and metric curve

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))

for ax, rung, row in zip(axes, ladder, results):
    evidence = []
    for item in rung["items"][:5]:
        evidence.append(vectorize_sentiment(item["text"]) * w)
    matrix = np.vstack(evidence)
    ax.imshow(matrix, aspect="auto", cmap="coolwarm", vmin=-2, vmax=2)
    ax.set_title(rung["rung"])
    ax.set_xticks(range(len(VOCAB)))
    ax.set_xticklabels(VOCAB, rotation=90, fontsize=7)
    ax.set_yticks([])

plt.suptitle("Token evidence heatmaps by rung")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot([row["rung"] for row in results], [row["accuracy"] for row in results], marker="o")
plt.ylim(0, 1.05)
plt.ylabel("accuracy")
plt.xlabel("rung")
plt.title("Accuracy vs negation and domain complexity")
plt.grid(True)
plt.show()


## Pitfall on D5: accuracy-only and vocabulary drift

Accuracy can hide asymmetric errors, and a serving-time vectorizer that changes feature order makes learned weights multiply the wrong evidence. We reproduce both on D5, then fix them with a frozen vectorizer and threshold inspection.

In [ ]:

d5 = ladder[-1]["items"]
y_true = [item["label"] for item in d5]

frozen_preds = []
for item in d5:
    x = vectorize_sentiment(item["text"], vocab=VOCAB)
    frozen_preds.append(sentiment_logreg(x, w, tau=0.5)[2])

shifted_vocab = list(reversed(VOCAB))
drifted_preds = []
for item in d5:
    x = vectorize_sentiment(item["text"], vocab=shifted_vocab)
    drifted_preds.append(sentiment_logreg(x, w, tau=0.5)[2])

frozen_acc = accuracy_score(y_true, frozen_preds)
drifted_acc = accuracy_score(y_true, drifted_preds)
precision, recall, f1 = precision_recall_f1(y_true, frozen_preds)

print("frozen accuracy", round(frozen_acc, 3))
print("drifted accuracy", round(drifted_acc, 3))
print("frozen precision recall f1", round(precision, 3), round(recall, 3), round(f1, 3))
assert frozen_acc >= drifted_acc


## Evaluate it + Practice

- Metric: classification accuracy on each rung, compared with the no-skill majority-class baseline.
- Sanity check: the lesson logit is exactly `2.4`, and `not good` scores below `good`.
- Ablation: remove the `not_good` composition feature and D3-D5 should drop.
- Failure signals: vocabulary drift, uncalibrated probabilities, and precision or recall collapse.

Practice: change the threshold to favor recall on D5 and report precision, recall, and F1.

Practice: remove `not_good` from the vocabulary and measure the error caused by unigram-only counting.

Practice: add one neutral class indicator and describe why binary accuracy is no longer enough.